# Rabin-Karp Algorithm

A hash-based string matching algorithm that uses rolling hash for efficient pattern searching.

## Key Insight

**If `hash(window) ≠ hash(pattern)`, there is definitely no match.**

This allows us to skip expensive character-by-character comparisons for most positions. We only need to verify when hashes match (which might be a collision).

## Topics Covered

1. **Hash-based Pattern Matching** - compare hashes instead of character strings
2. **Rolling Hash** - update hash in O(1) when sliding window
3. **Handling Collisions** - hash match doesn't guarantee string match
4. **Multiple Pattern Search** - where Rabin-Karp excels

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook builds algorithmic thinking used to reason about performance and correctness in computational biology tools.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Big-O is asymptotic guidance; constant factors still matter in practice.
- Correctness comes before optimization: verify edge cases before performance tuning.
- Data structure choice often dominates speed more than micro-optimizations.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Knuth-Morris-Pratt (KMP) Algorithm](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/02_kmp_algorithm.ipynb) | [Next: DFA-Based Pattern Matching →](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/04_dfa_matching.ipynb)

---

---

## 1. Hash-based Pattern Matching

### The Basic Idea

Instead of comparing strings character by character, we compare their hash values:

```
Naive approach:     Compare m characters → O(m) per position
Hash-based:         Compare 2 integers  → O(1) per position
```

### Why It Works

- **Different strings usually have different hashes** → Quick rejection
- **Same hash might mean collision** → Need verification
- **Net effect**: Dramatically fewer comparisons on average

---

## 2. Rolling Hash Concept

The magic of Rabin-Karp is the **rolling hash** - updating the hash in O(1) as we slide the window.

```
Text: A B C D E F G
      └───┴───┘
      window = "ABCD"
      hash = h("ABCD")
      
Slide window:
Text: A B C D E F G
        └───┴───┘
        window = "BCDE"
        
Instead of recomputing h("BCDE") from scratch:
  new_hash = roll(old_hash, remove='A', add='E')

Rolling: O(1) instead of O(m)!
```

### Rolling Hash Formula

We treat a string as a number in base `d` (typically 256 for ASCII):

```
h(s[i..i+m-1]) = s[i]*d^(m-1) + s[i+1]*d^(m-2) + ... + s[i+m-1]*d^0
```

To roll from position `i` to `i+1`:

```
h(s[i+1..i+m]) = d * (h(s[i..i+m-1]) - s[i]*d^(m-1)) + s[i+m]
```

**In plain English:**
1. Subtract contribution of outgoing character (scaled by d^(m-1))
2. Multiply by d (shift all positions left)
3. Add incoming character

```
Base d = 256 (ASCII), prime q = 101

Pattern: "ABC"  →  h = (65*256² + 66*256 + 67) mod 101

Text: "XYZABC"
Window "XYZ": h = (88*256² + 89*256 + 90) mod 101

Roll to "YZA":
  1. Remove 'X': (h - 88*256²) mod 101
  2. Shift left: result * 256
  3. Add 'A': result + 65
  4. Mod: result mod 101

Formula: h_new = (d * (h_old - char_out * d^(m-1)) + char_in) mod q
```

### Why Use Modular Arithmetic?

Without modulo, hash values would grow exponentially:

```
Pattern length m = 10, base d = 256
Hash could be as large as: 256^10 = 1,208,925,819,614,629,174,706,176
```

**Problems without modulo:**
- Integer overflow
- Arithmetic becomes expensive (arbitrary precision)
- Comparison becomes O(m) for big integers

**Benefits of modular arithmetic:**
- Hash values stay bounded (0 to q-1)
- All arithmetic is O(1)
- Choose prime q to minimize collisions

**Why prime q?**
- Distributes hash values more uniformly
- Reduces patterns in the hash function
- Common choice: large prime like 101, 1000000007

---

## 3. Handling Collisions (Spurious Hits)

A **hash collision** occurs when different strings have the same hash.

```
Text:    "AABAACAADAAB"
Pattern: "AAB"
Pattern hash = 65*256² + 65*256 + 66 = 4276546 mod 101 = 80

Position 0: "AAB" → hash = 80 → MATCH! Verify: "AAB" = "AAB" ✓
Position 1: "ABA" → hash = 82 → no match
Position 2: "BAA" → hash = 95 → no match
Position 3: "AAC" → hash = 81 → no match
Position 4: "ACA" → hash = 72 → no match
...

Hash collision example:
Position 5: "CAA" → hash = 80 → HASH MATCH! Verify: "CAA" ≠ "AAB" ✗
                                 (spurious hit - false positive)
```

### The Rule

**Always verify on hash match!**

```python
if hash_window == hash_pattern:
    if text[i:i+m] == pattern:  # Actual comparison
        found_match(i)
```

This verification costs O(m) but happens rarely with a good hash function.

---

## 4. Search Process Visualization

```
Text:    [A][B][R][A][C][A][D][A][B][R][A]
Pattern: [A][B][R][A]

Step 1: Compute pattern hash
        h("ABRA") = (65·256³ + 66·256² + 82·256 + 65) mod q

Step 2: Compute first window hash
        h(text[0:4]) = h("ABRA") 

Step 3: Compare and slide

  pos=0: [A][B][R][A] C  A  D  A  B  R  A
         └────────┘
         hash = h_p → MATCH! Verify ✓ → Found at 0

  pos=1:  A [B][R][A][C] A  D  A  B  R  A
            └────────┘
         hash = roll(h, -'A', +'C') → ≠ h_p

  pos=2:  A  B [R][A][C][A] D  A  B  R  A
               └────────┘
         hash = roll(...) → ≠ h_p

  ...continue rolling...

  pos=7:  A  B  R  A  C  A  D [A][B][R][A]
                              └────────┘
         hash = h_p → MATCH! Verify ✓ → Found at 7
```

---

## Complexity Analysis

| Case | Time Complexity | Notes |
|------|-----------------|-------|
| **Average/Expected** | O(n + m) | Few hash collisions |
| **Worst case** (many collisions) | O(n × m) | Every position needs verification |
| **Space** | O(1) | Just storing hash values |

### When Does Worst Case Occur?

```
Text:    "AAAAAAAAAA"
Pattern: "AAA"

Every window has the same hash → verify at every position!
```

**With a good hash function and prime modulus, worst case is rare in practice.**

---

## Implementation

In [ ]:
def rabin_karp_search(text: str, pattern: str, d: int = 256, q: int = 101) -> list[int]:
    """
    Find all occurrences of pattern in text using Rabin-Karp algorithm.
    
    Uses rolling hash to achieve O(n+m) average time complexity.
    
    Parameters
    ----------
    text : str
        The text to search in.
    pattern : str
        The pattern to search for.
    d : int, optional
        Base for the hash function (default 256 for ASCII).
    q : int, optional
        Prime modulus to prevent overflow (default 101).
        Larger primes reduce collision probability.
    
    Returns
    -------
    list[int]
        List of starting indices where pattern occurs in text.
    
    Examples
    --------
    >>> rabin_karp_search("AABAACAADAABAABA", "AABA")
    [0, 9, 12]
    >>> rabin_karp_search("abcdef", "xyz")
    []
    
    Notes
    -----
    Rolling hash formula:
        h(s[i+1..i+m]) = (d * (h(s[i..i+m-1]) - s[i] * d^(m-1)) + s[i+m]) mod q
    
    The algorithm verifies actual string equality on hash matches to handle
    collisions (spurious hits).
    """
    n = len(text)
    m = len(pattern)
    matches = []
    
    # Edge cases
    if m == 0 or m > n:
        return matches
    
    # Precompute d^(m-1) mod q for rolling hash
    # This is the multiplier for the leftmost character
    h = 1
    for _ in range(m - 1):
        h = (h * d) % q
    
    # Compute initial hash values
    pattern_hash = 0
    window_hash = 0
    
    for i in range(m):
        pattern_hash = (d * pattern_hash + ord(pattern[i])) % q
        window_hash = (d * window_hash + ord(text[i])) % q
    
    # Slide pattern over text
    for i in range(n - m + 1):
        # Check if hashes match
        if pattern_hash == window_hash:
            # Verify actual string match (handle collisions)
            if text[i:i+m] == pattern:
                matches.append(i)
        
        # Compute hash for next window using rolling hash
        if i < n - m:
            # Remove leading character, add trailing character
            window_hash = (d * (window_hash - ord(text[i]) * h) + ord(text[i + m])) % q
            
            # Handle negative modulo (Python handles this, but be explicit)
            if window_hash < 0:
                window_hash += q
    
    return matches

In [ ]:
# Test the basic implementation
print("Basic tests:")
print(f"  'AABAACAADAABAABA' contains 'AABA' at: {rabin_karp_search('AABAACAADAABAABA', 'AABA')}")
print(f"  'abcdef' contains 'xyz' at: {rabin_karp_search('abcdef', 'xyz')}")
print(f"  'ABRACADABRA' contains 'ABRA' at: {rabin_karp_search('ABRACADABRA', 'ABRA')}")

---

## Step-by-Step Hash Computation Example

In [ ]:
def demonstrate_hash_computation(text: str, pattern: str, d: int = 256, q: int = 101):
    """
    Demonstrate step-by-step hash computation in Rabin-Karp.
    """
    n, m = len(text), len(pattern)
    
    print(f"Text:    '{text}'")
    print(f"Pattern: '{pattern}' (length m={m})")
    print(f"Base d={d}, Prime q={q}")
    print("=" * 60)
    
    # Compute d^(m-1) mod q
    h = pow(d, m-1, q)
    print(f"\nd^(m-1) mod q = {d}^{m-1} mod {q} = {h}")
    
    # Compute pattern hash step by step
    print(f"\n--- Pattern Hash Computation ---")
    pattern_hash = 0
    for i, char in enumerate(pattern):
        old_hash = pattern_hash
        pattern_hash = (d * pattern_hash + ord(char)) % q
        print(f"  char '{char}' (ASCII {ord(char)}): ({d} × {old_hash} + {ord(char)}) mod {q} = {pattern_hash}")
    print(f"  Pattern hash = {pattern_hash}")
    
    # Compute first window hash
    print(f"\n--- Initial Window Hash ---")
    window_hash = 0
    for i in range(m):
        old_hash = window_hash
        window_hash = (d * window_hash + ord(text[i])) % q
        print(f"  char '{text[i]}' (ASCII {ord(text[i])}): ({d} × {old_hash} + {ord(text[i])}) mod {q} = {window_hash}")
    print(f"  Window[0:{m}] = '{text[0:m]}', hash = {window_hash}")
    
    # Rolling hash demonstration
    print(f"\n--- Rolling Hash in Action ---")
    for i in range(min(5, n - m)):  # Show first 5 rolls
        old_hash = window_hash
        char_out = text[i]
        char_in = text[i + m]
        
        # Rolling hash formula
        window_hash = (d * (window_hash - ord(char_out) * h) + ord(char_in)) % q
        if window_hash < 0:
            window_hash += q
        
        match_status = "HASH MATCH!" if window_hash == pattern_hash else ""
        print(f"  Roll: remove '{char_out}', add '{char_in}'")
        print(f"        Window[{i+1}:{i+1+m}] = '{text[i+1:i+1+m]}', hash = {window_hash} {match_status}")
        
        if match_status:
            actual_match = text[i+1:i+1+m] == pattern
            print(f"        Verify: '{text[i+1:i+1+m]}' == '{pattern}' → {actual_match}")

# Demonstrate with a simple example
demonstrate_hash_computation("ABRACADABRA", "ABR")

---

## Demonstrating Spurious Hits (Collisions)

In [ ]:
def find_spurious_hits(text: str, pattern: str, d: int = 256, q: int = 11):
    """
    Find and display spurious hits (hash collisions that aren't actual matches).
    Uses small prime q to increase collision probability for demonstration.
    """
    n, m = len(text), len(pattern)
    h = pow(d, m-1, q)
    
    # Compute pattern hash
    pattern_hash = 0
    for char in pattern:
        pattern_hash = (d * pattern_hash + ord(char)) % q
    
    # Compute first window hash
    window_hash = 0
    for i in range(m):
        window_hash = (d * window_hash + ord(text[i])) % q
    
    print(f"Text: '{text}'")
    print(f"Pattern: '{pattern}' (hash = {pattern_hash} with small prime q={q})")
    print("=" * 60)
    
    true_matches = []
    spurious_hits = []
    
    for i in range(n - m + 1):
        window = text[i:i+m]
        
        if window_hash == pattern_hash:
            if window == pattern:
                true_matches.append((i, window))
                print(f"  pos {i}: '{window}' hash={window_hash} → TRUE MATCH ✓")
            else:
                spurious_hits.append((i, window))
                print(f"  pos {i}: '{window}' hash={window_hash} → SPURIOUS HIT ✗ (collision!)")
        
        # Roll hash
        if i < n - m:
            window_hash = (d * (window_hash - ord(text[i]) * h) + ord(text[i + m])) % q
            if window_hash < 0:
                window_hash += q
    
    print("\nSummary:")
    print(f"  True matches: {len(true_matches)}")
    print(f"  Spurious hits: {len(spurious_hits)}")
    return true_matches, spurious_hits

# Use small prime to force collisions
find_spurious_hits("ABCDEFGHIJKLMNOPQRSTUVWXYZABC", "ABC", q=11)

In [ ]:
# Another collision example
print("\nWith very small prime q=7:")
find_spurious_hits("AABAACAADAABAABCAAB", "AAB", q=7)

---

## Multiple Pattern Search

**This is where Rabin-Karp really shines!**

For searching k patterns of the same length:
- Naive: O(k × n × m)
- Rabin-Karp: O(n + k × m) average

We compute one rolling hash over the text, and check against all pattern hashes.

In [ ]:
def rabin_karp_multi(text: str, patterns: list[str], d: int = 256, q: int = 101) -> dict[str, list[int]]:
    """
    Search for multiple patterns of the same length using Rabin-Karp.
    
    This is the key advantage of Rabin-Karp: searching for k patterns
    only requires one pass through the text with one rolling hash.
    
    Parameters
    ----------
    text : str
        The text to search in.
    patterns : list[str]
        List of patterns to search for (must all have same length).
    d : int, optional
        Base for hash function.
    q : int, optional
        Prime modulus.
    
    Returns
    -------
    dict[str, list[int]]
        Dictionary mapping each pattern to its list of match positions.
    
    Raises
    ------
    ValueError
        If patterns have different lengths.
    """
    if not patterns:
        return {}
    
    # Verify all patterns have same length
    m = len(patterns[0])
    if not all(len(p) == m for p in patterns):
        raise ValueError("All patterns must have the same length")
    
    n = len(text)
    results = {p: [] for p in patterns}
    
    if m == 0 or m > n:
        return results
    
    # Compute hash for each pattern and store in a hash table
    # Multiple patterns can have the same hash (use list)
    pattern_hashes = {}  # hash -> [patterns with this hash]
    
    for pattern in patterns:
        p_hash = 0
        for char in pattern:
            p_hash = (d * p_hash + ord(char)) % q
        
        if p_hash not in pattern_hashes:
            pattern_hashes[p_hash] = []
        pattern_hashes[p_hash].append(pattern)
    
    # Compute d^(m-1) mod q
    h = pow(d, m-1, q)
    
    # Compute first window hash
    window_hash = 0
    for i in range(m):
        window_hash = (d * window_hash + ord(text[i])) % q
    
    # Single pass through text
    for i in range(n - m + 1):
        # Check if window hash matches any pattern hash
        if window_hash in pattern_hashes:
            window = text[i:i+m]
            # Verify against all patterns with this hash
            for pattern in pattern_hashes[window_hash]:
                if window == pattern:
                    results[pattern].append(i)
        
        # Roll hash
        if i < n - m:
            window_hash = (d * (window_hash - ord(text[i]) * h) + ord(text[i + m])) % q
            if window_hash < 0:
                window_hash += q
    
    return results

In [ ]:
# Example: Search for multiple DNA motifs
dna_sequence = "ACTGACTGCACTGACTGAACTGACTG"
motifs = ["ACTG", "GACT", "CTGA", "TGAC"]

print(f"DNA Sequence: {dna_sequence}")
print(f"Searching for motifs: {motifs}")
print("=" * 50)

results = rabin_karp_multi(dna_sequence, motifs)
for motif, positions in results.items():
    print(f"  '{motif}' found at positions: {positions}")

In [ ]:
# Plagiarism detection example
document = "the quick brown fox jumps over the lazy dog the quick brown cat"
suspicious_phrases = ["the quick", "brown fox", "brown cat", "lazy dog"]

print(f"Document: '{document}'")
print(f"\nSearching for suspicious phrases of length 9:")

# All phrases must be same length for multi-pattern search
phrases_9 = [p for p in suspicious_phrases if len(p) == 9]
results = rabin_karp_multi(document, phrases_9)

for phrase, positions in results.items():
    if positions:
        print(f"  Found '{phrase}' at positions: {positions}")

---

## Comparison with KMP

In [ ]:
def kmp_search(text: str, pattern: str) -> list[int]:
    """
    KMP algorithm for comparison.
    """
    def compute_lps(pattern):
        m = len(pattern)
        lps = [0] * m
        length = 0
        i = 1
        while i < m:
            if pattern[i] == pattern[length]:
                length += 1
                lps[i] = length
                i += 1
            else:
                if length != 0:
                    length = lps[length - 1]
                else:
                    lps[i] = 0
                    i += 1
        return lps
    
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []
    
    lps = compute_lps(pattern)
    matches = []
    
    i = j = 0
    while i < n:
        if pattern[j] == text[i]:
            i += 1
            j += 1
        
        if j == m:
            matches.append(i - j)
            j = lps[j - 1]
        elif i < n and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    
    return matches

In [ ]:
import time

def compare_algorithms(text: str, pattern: str, iterations: int = 1000):
    """
    Compare Rabin-Karp and KMP performance.
    """
    # Verify both give same results
    rk_result = rabin_karp_search(text, pattern)
    kmp_result = kmp_search(text, pattern)
    assert rk_result == kmp_result, f"Results differ! RK: {rk_result}, KMP: {kmp_result}"
    
    # Time Rabin-Karp
    start = time.perf_counter()
    for _ in range(iterations):
        rabin_karp_search(text, pattern)
    rk_time = time.perf_counter() - start
    
    # Time KMP
    start = time.perf_counter()
    for _ in range(iterations):
        kmp_search(text, pattern)
    kmp_time = time.perf_counter() - start
    
    print(f"Text length: {len(text)}, Pattern length: {len(pattern)}")
    print(f"Matches found: {len(rk_result)} at positions {rk_result[:5]}{'...' if len(rk_result) > 5 else ''}")
    print(f"Rabin-Karp: {rk_time*1000:.2f} ms ({iterations} iterations)")
    print(f"KMP:        {kmp_time*1000:.2f} ms ({iterations} iterations)")
    print(f"Ratio:      RK is {kmp_time/rk_time:.2f}x {'faster' if rk_time < kmp_time else 'slower'} than KMP")

print("Test 1: Short text, multiple matches")
compare_algorithms("AABAACAADAABAABAABAAB", "AAB")

print("\nTest 2: Longer text, few matches")
text = "abcdefghij" * 1000 + "xyz" + "abcdefghij" * 1000
compare_algorithms(text, "xyz")

print("\nTest 3: Worst case for Rabin-Karp (many collisions potential)")
compare_algorithms("a" * 10000, "aaa")

### When to Use Which Algorithm?

| Scenario | Best Choice | Why |
|----------|-------------|-----|
| Single pattern search | KMP | Guaranteed O(n+m), no verification needed |
| Multiple patterns (same length) | **Rabin-Karp** | One pass for all patterns |
| Plagiarism detection | **Rabin-Karp** | Many phrases to search |
| DNA motif search | **Rabin-Karp** | Multiple motifs |
| Real-time single pattern | KMP | Predictable performance |

---

## Original Implementation Reference

The original homework implementation used a DNA-specific encoding:

In [ ]:
# Original implementation from Algorithms_HW7.ipynb
# Uses base-10 with DNA alphabet {a:1, t:2, g:3, c:4}

def make_hash_dna(string: str) -> int:
    """Original DNA hash function."""
    d = {"a": 1, "t": 2, "g": 3, "c": 4}
    out = 0
    for char in string:
        out = out * 10 + d[char]
    return out


def rabin_karp_dna(pattern: str, text: str) -> list[int]:
    """
    Original Rabin-Karp implementation for DNA sequences.
    
    Note: This version doesn't use modular arithmetic,
    which could cause issues with very long patterns.
    """
    if len(pattern) > len(text) or len(pattern) == 0 or len(text) == 0:
        return []
    
    m = len(pattern)
    pattern_hash = make_hash_dna(pattern)
    text_hash = make_hash_dna(text[0:m])
    
    matches = []
    if text_hash == pattern_hash:
        matches.append(0)
    
    for i in range(1, len(text) - m + 1):
        # Rolling hash: remove first digit, add new digit
        text_hash = text_hash % (10 ** (m - 1)) * 10 + make_hash_dna(text[m + i - 1])
        if text_hash == pattern_hash:
            matches.append(i)
    
    return matches

# Test original implementation
print("Original DNA implementation tests:")
print(f"  rabin_karp_dna('ct', 'actgc') = {rabin_karp_dna('ct', 'actgc')}")
print(f"  rabin_karp_dna('ac', 'actgc') = {rabin_karp_dna('ac', 'actgc')}")
print(f"  rabin_karp_dna('ctg', 'actgc') = {rabin_karp_dna('ctg', 'actgc')}")
print(f"  rabin_karp_dna('actgc', 'actgcactgc') = {rabin_karp_dna('actgc', 'actgcactgc')}")

---

## Summary

### Key Takeaways

1. **Hash-based matching**: Compare hash values instead of strings
2. **Rolling hash**: O(1) hash update when sliding window
3. **Verification required**: Hash match ≠ string match (handle collisions)
4. **Multiple patterns**: Main advantage over KMP

### Complexity Recap

| Metric | Value |
|--------|-------|
| Average Time | O(n + m) |
| Worst Time | O(n × m) |
| Space | O(1) |
| Multi-pattern (k patterns) | O(n + k×m) average |

### Best For

- Plagiarism detection
- Multiple pattern search
- DNA/protein motif finding
- When patterns share same length

## Alternative Implementation (Kodomo)

This alternative uses a simpler hash based on digit concatenation. It's less robust than the polynomial hash but illustrates the core rolling hash concept.

Each character is mapped to a single digit via `ord(c) - 96` (so `'a'` → 1, `'b'` → 2, …), and those digits are concatenated to form an integer. Rolling is done by dropping the leading digit with modulo arithmetic and appending the new character's digit. Because digits can exceed 9 for characters beyond `'j'`, the hash is not collision-free, but the algorithm still works correctly because every candidate match is verified by character comparison.

In [ ]:
def make_hash(string: str, rehash: list | None = None) -> int:
    """Digit-concatenation hash (Kodomo style).

    When rehash is None: hash the full string.
    When rehash=[old_hash, pattern_len]: roll the hash one step forward
    by appending the single character in `string`.
    """
    if rehash is None:
        out_str = ""
        for ch in string:
            out_str += str(ord(ch) - 96)[0]
        return int(out_str)
    else:
        old_hash, pattern_len = rehash[0], rehash[1]
        modulus = int("1" + "0" * (pattern_len - 1)) if pattern_len > 1 else 1
        out_str = str(old_hash % modulus) + str(ord(string) - 96)[0]
        return int(out_str)


def rabin_karp_kodomo(pattern: str, text: str) -> int:
    """Return the start index of the first match, or -1.

    Uses digit-concatenation rolling hash (works for lowercase ASCII input).
    """
    if not pattern or not text or len(pattern) > len(text):
        return -1

    pattern_hash = make_hash(pattern)
    text_hash = make_hash(text[:len(pattern)])

    for i in range(len(text) - len(pattern) + 1):
        if text_hash == pattern_hash:
            return i
        if i + len(pattern) < len(text):
            text_hash = make_hash(text[i + len(pattern)],
                                  rehash=[text_hash, len(pattern)])
    return -1


# Note: hash function maps lowercase letters a-z to digits 1-26.
# Uppercase or non-letter characters produce unexpected offsets.
print(rabin_karp_kodomo("gc", "aaaaatgcatgc"))    # 5
print(rabin_karp_kodomo("atgc", "ttttatgcgggg"))  # 4
print(rabin_karp_kodomo("aaaa", "tttgatgcgggg"))  # -1

---

## Exercises

### Exercise 1: Motif Search with Hash Comparison Counting (*)

Rolling hash lets Rabin-Karp skip character comparisons for most windows. Understanding the ratio of hash comparisons to actual character comparisons reveals how well the algorithm performs in practice.

**Task:** Instrument `rabin_karp_search` to count separately:
1. **Hash comparisons**: how many times `window_hash == pattern_hash` is tested
2. **Character comparisons**: how many times an actual string equality check (`text[i:i+m] == pattern`) is performed

Search for motif `"ATCG"` in the DNA sequence below and report both counts. Compare with the theoretical minimum (one hash comparison per window position).

In [ ]:
import random
random.seed(42)
DNA_LONG = ''.join(random.choice('ACGT') for _ in range(1000))
# Plant the motif at a few known positions
_dna = list(DNA_LONG)
for pos in [100, 300, 600, 850]:
    for i, c in enumerate("ATCG"):
        _dna[pos + i] = c
DNA_LONG = ''.join(_dna)

MOTIF = "ATCG"


def rabin_karp_counted(
    text: str, pattern: str, d: int = 256, q: int = 101
) -> tuple[list[int], int, int]:
    """
    Rabin-Karp search that counts hash and character comparisons.

    Args:
        text: Text to search in
        pattern: Pattern to search for
        d: Hash base
        q: Prime modulus

    Returns:
        Tuple of (match_positions, hash_comparisons, char_comparisons)
    """
    # YOUR CODE HERE
    pass


# positions, hash_cmps, char_cmps = rabin_karp_counted(DNA_LONG, MOTIF)
# print(f"Motif '{MOTIF}' found at: {positions}")
# print(f"Hash comparisons:      {hash_cmps}  (one per window = {len(DNA_LONG) - len(MOTIF) + 1})")
# print(f"Character comparisons: {char_cmps}  (only on hash hits)")
# print(f"Savings: {(1 - char_cmps / hash_cmps) * 100:.1f}% of positions skipped")

### Exercise 2: Multi-Pattern Rabin-Karp for Simultaneous Motif Search (**)

Rabin-Karp's key advantage over KMP is that k patterns of the same length can be searched in a single pass. This is directly useful when scanning a genome for many motifs at once (e.g., all known splice-site sequences).

**Task:** Implement `multi_pattern_search_rk(reference, patterns)` using the multi-pattern variant (`rabin_karp_multi`). Then compare its wall-clock time against running `rabin_karp_search` once per pattern. Use `time.perf_counter` for timing.

Report: total matches found, multi-pattern time, single-pattern total time, and the speedup ratio.

In [ ]:
import time
import random
random.seed(99)

REFERENCE = ''.join(random.choice('ACGT') for _ in range(5000))

SPLICE_MOTIFS = {
    "donor_GT":    "GTAAGT",
    "donor_GC":    "GCAAGT",
    "acceptor_AG": "TTTCAG",
    "acceptor_AA": "TTTAAG",
    "branch_A":    "CTAACT",
    "polypyrim":   "TTTTTC",
}


def multi_pattern_search_rk(
    reference: str, patterns: dict[str, str]
) -> dict[str, list[int]]:
    """
    Search for multiple same-length motifs in a single Rabin-Karp pass.

    Args:
        reference: The DNA sequence to search
        patterns: Dict mapping motif name to sequence (all must be same length)

    Returns:
        Dict mapping motif name to list of match positions
    """
    # YOUR CODE HERE
    pass


# # Time multi-pattern search
# t0 = time.perf_counter()
# multi_results = multi_pattern_search_rk(REFERENCE, SPLICE_MOTIFS)
# t_multi = time.perf_counter() - t0
#
# # Time individual searches
# t0 = time.perf_counter()
# single_results = {name: rabin_karp_search(REFERENCE, seq)
#                   for name, seq in SPLICE_MOTIFS.items()}
# t_single = time.perf_counter() - t0
#
# print("Multi-pattern results:")
# for name, positions in multi_results.items():
#     print(f"  {name}: {len(positions)} hits")
#
# results_match = all(multi_results[n] == single_results[n] for n in SPLICE_MOTIFS)
# print(f"\nResults match single-pattern: {results_match}")
# print(f"Multi-pattern time:   {t_multi*1000:.3f} ms")
# print(f"Single-pattern total: {t_single*1000:.3f} ms")
# if t_single > 0:
#     print(f"Speedup: {t_single/t_multi:.2f}x")

---

[← Previous: Knuth-Morris-Pratt (KMP) Algorithm](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/02_kmp_algorithm.ipynb) | [Next: DFA-Based Pattern Matching →](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/04_dfa_matching.ipynb)